<a href="https://colab.research.google.com/github/JuanSamaniego02/curso-ia-para-economia/blob/main/2_Taller_Apriori_AlanRios_JuanSamaniego.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/LinaMariaCastro/curso-ia-para-economia/blob/main/clases/4_Aprendizaje_no_supervisado/2_Taller_Apriori.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# **Inteligencia Artificial con Aplicaciones en Economía I**

- 👩‍🏫 **Profesora:** [Lina María Castro](https://www.linkedin.com/in/lina-maria-castro)  
- 📧 **Email:** [lmcastroco@gmail.com](mailto:lmcastroco@gmail.com)  
- 🎓 **Universidad:** Universidad Externado de Colombia - Facultad de Economía

# **Taller: Análisis de Patrones de Consumo Internacional con Apriori**

**IMPORTANTE**: Guarda una copia de este notebook en tu Google Drive o computador.

**Taller en grupos de 3**

**Nombres estudiantes:**

- Alan Rios
- Juan Esteban Samaniego
-

**Forma de entrega:**

- Nombrar el archivo de la siguiente forma:“Taller_Apriori_apellidos.ipynb”.
- Suba el Jupyter Notebook a su cuenta en Github y envíe el link en el siguiente Forms: https://forms.cloud.microsoft/r/qERdEpXpmx.

**IMPORTANTE:** No se recibirán talleres en Google Colab, el notebook debe estar subido en Github.

**Plazo de entrega:**

21 de abril de 2026, máximo a las 11:59 p.m. Tenga en cuenta que luego de esa hora el formulario en forms se cierra. El Jupupyter Notebook también debe quedar subido en Github antes de esa hora.

**Instrucciones Generales:**

Completa el código en las celdas marcadas con `### TU CÓDIGO AQUÍ ###`. Puedes añadir más celdas si lo requieres.

**Caso de Estudio: Consultoría para Global Retail Inc.**

**Contexto:** Una firma multinacional de e-commerce, "Global Retail Inc.", te ha contratado como consultor de datos. La empresa opera en múltiples países y ha notado que sus ventas y la efectividad de sus campañas de marketing varían significativamente entre regiones. Su hipótesis es que los patrones de compra y las asociaciones de productos son diferentes en cada mercado.

**Tu Misión:** Analizar el historial de transacciones de la empresa para descubrir y comparar las reglas de asociación de productos para dos de sus mercados más importantes en Latinoamérica: México y Colombia. Tu objetivo final es entregar recomendaciones de negocio accionables (ej. estrategias de cross-selling, promociones personalizadas) basadas en los patrones de consumo que descubras en cada país.

**Dataset:** Encuentra mayor información en el archivo "diccionario_alimentos_retail_top30.xlsx".

## Ejercicio 1: Configuración Inicial, Carga y Exploración de Datos

1.1 Importa las librerías necesarias

In [ ]:
import os
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Configuraciones de visualización
pd.options.display.max_columns = None
pd.options.display.float_format = '{:,.2f}'.format

1.2 Carga el dataset "alimentos_retail_top30.csv" que se encuentra en el repositorio del curso, carpeta "datasets". El dataframe debe llamarse "df".

In [ ]:

df = "/content/alimentos_retail_top30.csv"
df = pd.read_csv("alimentos_retail_top30.csv")

In [ ]:
# Debe ser (6899, 8)
print("Dimensiones del DataFrame:")
print(df.shape)
df.head()

Dimensiones del DataFrame:
(6899, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536000,94537,HARINA DE MAÍZ,5,2023-01-07 01:09:00,2.76,"17,452.00",Colombia
1,536000,87297,QUESO MUZZARELLA,2,2023-01-07 01:09:00,4.69,"17,779.00",Colombia
2,536001,94537,HARINA DE MAÍZ,4,2023-01-07 11:51:00,2.76,"14,933.00",Colombia
3,536001,87297,QUESO MUZZARELLA,3,2023-01-07 11:51:00,4.69,"14,957.00",Colombia
4,536002,26907,CAFÉ,4,2023-01-02 01:54:00,2.36,"15,202.00",Colombia


In [ ]:
print("\nInformación general del DataFrame:")
df.info()


Información general del DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6899 entries, 0 to 6898
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   InvoiceNo    6899 non-null   object 
 1   StockCode    6899 non-null   int64  
 2   Description  6899 non-null   object 
 3   Quantity     6899 non-null   int64  
 4   InvoiceDate  6899 non-null   object 
 5   UnitPrice    6899 non-null   float64
 6   CustomerID   6879 non-null   float64
 7   Country      6899 non-null   object 
dtypes: float64(2), int64(2), object(4)
memory usage: 431.3+ KB


1.3 Revisa si hay valores nulos en alguna columna y cuántos son

In [ ]:

df.isnull().sum()

,0
InvoiceNo,0
StockCode,0
Description,0
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,20
Country,0


1.4 Genera las estadísticas descriptivas de las variables numéricas

In [ ]:

df.describe([])


,StockCode,Quantity,UnitPrice,CustomerID
count,"6,899.00","6,899.00","6,899.00","6,879.00"
mean,"55,544.94",3.00,3.42,"15,024.12"
std,"25,875.73",1.43,1.06,"1,732.95"
min,"26,907.00",-5.00,1.65,"12,000.00"
50%,"42,889.00",3.00,3.39,"15,041.00"
max,"95,931.00",5.00,4.90,"17,999.00"


In [ ]:
df.select_dtypes(include='number').describe()

,StockCode,Quantity,UnitPrice,CustomerID
count,"6,899.00","6,899.00","6,899.00","6,879.00"
mean,"55,544.94",3.00,3.42,"15,024.12"
std,"25,875.73",1.43,1.06,"1,732.95"
min,"26,907.00",-5.00,1.65,"12,000.00"
25%,"31,048.00",2.00,2.36,"13,524.00"
50%,"42,889.00",3.00,3.39,"15,041.00"
75%,"87,297.00",4.00,4.44,"16,530.50"
max,"95,931.00",5.00,4.90,"17,999.00"


1.5 Observando las salidas del ejercicio anterior, ¿qué problemas potenciales identificas en las columnas CustomerID y Quantity?

## Ejercicio 2: Limpieza y Preprocesamiento de Datos

Los datos del mundo real rara vez son perfectos. Antes de cualquier análisis, debemos "sanear" nuestro dataset. Completa el código en cada paso según las instrucciones.

Crea un nuevo dataframe llamado "df_limpio" para los siguientes puntos.

2.1 **Manejo de Valores Nulos**: Las transacciones sin un CustomerID no son útiles para nosotros, ya que no podemos agrupar las compras de un cliente específico.

In [ ]:
# TAREA: Elimina todas las filas donde 'CustomerID' es nulo.

df_limpio = df.dropna(subset=['CustomerID'])

In [ ]:
# El tipo de dato de CustomerID debe ser entero

df_limpio['CustomerID'] = df_limpio['CustomerID'].astype(int)

2.2 **Limpieza de Descripciones de Productos** Las descripciones pueden tener espacios en blanco al inicio o al final que podrían hacer que un mismo producto se cuente como dos diferentes.

In [ ]:
# TAREA: # Verifica cuántas descripciones únicas hay.

df_limpio['Description'].nunique()

25

In [ ]:
# TAREA: Limpia la columna 'Description' eliminando espacios extra al inicio y al final.

df_limpio['Description'] = df_limpio['Description'].str.strip()

In [ ]:
# TAREA: Verifica cuántas descripciones únicas quedaron después de la limpieza.
df_limpio['Description'].nunique()

20

2.3 **Filtrado de Transacciones Anómalas**: Las facturas (InvoiceNo) que empiezan con 'C' indican una cancelación. Estas no son compras reales y deben ser eliminadas. Del mismo modo, las cantidades (Quantity) negativas representan devoluciones.

In [ ]:
# TAREA: Elimina las filas que correspondan a cancelaciones.
df_limpio['InvoiceNo'] = df_limpio['InvoiceNo'].astype(str).str.strip()
df_limpio = df_limpio[~df_limpio['InvoiceNo'].str.startswith('C')]

In [ ]:
# TAREA: Elimina las filas con cantidades negativas.
df_limpio = df_limpio[df_limpio['Quantity'] > 0]

In [ ]:
# Verifiquemos las dimensiones del DataFrame después de la limpieza. Debe ser (6864, 8)
df_limpio.shape

(6864, 8)

## Ejercicio 3: Análisis Comparativo por País

Ahora que los datos están limpios, vamos a segmentarlos y a aplicar el algoritmo Apriori para encontrar los patrones de compra en México y Colombia.

**Preparación de la Cesta de Mercado (Función)**

La siguiente función toma un dataframe, lo agrupa por factura y descripción, y lo transforma en el formato de matriz binaria que necesita el algoritmo Apriori. Estudia esta función, no necesitas modificarla.

In [ ]:
def preparar_cesta(dataframe, pais):
    """Filtra por país y prepara la matriz de transacciones."""

    # Filtrar por el país de interés
    df_pais = dataframe[dataframe['Country'] == pais]

    # Crear la cesta: agrupar productos por factura
    cesta = (df_pais.groupby(['InvoiceNo', 'Description'])['Quantity']
             .sum().unstack().reset_index().fillna(0)
             .set_index('InvoiceNo'))

    # Convertir todas las cantidades positivas a 1 y todo lo demás a 0
    cesta_encoded = (cesta > 0).astype(int)

    return cesta_encoded

3.1 Análisis para México

In [ ]:
# TAREA: Usa la función preparar_cesta para obtener la matriz de transacciones de México. Almacena el resultado en la variable cesta_mx.
### TU CÓDIGO AQUÍ ###
cesta_mx = preparar_cesta(df_limpio,"México")
cesta_mx

Description,AGUACATE,CEBOLLA,CHILE JALAPEÑO,CILANTRO,FRIJOL NEGRO,LIMÓN,QUESO FRESCO,TOMATE,TORTILLAS DE MAÍZ,TOTOPOS
InvoiceNo,,,,,,,,,,
537000,0,0,0,0,1,0,0,0,1,0
537001,0,1,1,1,0,0,0,1,0,0
537002,0,0,0,0,1,0,0,1,1,0
537003,0,1,1,1,0,0,0,1,0,0
537004,0,0,0,1,0,0,1,1,1,0
...,...,...,...,...,...,...,...,...,...,...
537995,0,0,0,0,1,0,0,0,1,0
537996,0,1,1,1,0,0,0,1,0,0
537997,0,0,0,0,0,0,0,1,0,1


In [ ]:
# TAREA: Aplica el algoritmo apriori para encontrar itemsets con un soporte mínimo de 2%.
# Almacena el resultado en la variable frequent_itemsets_mx.
# Muestra los 10 itemsets con el soporte más alto.
frequent_itemsets_mx  = apriori(cesta_mx, min_support = 0.02, use_colnames=True)
frequent_itemsets_mx.sort_values(by='support', ascending=False).head(10)

,support,itemsets
2,0.42,(CHILE JALAPEÑO)
7,0.41,(TOMATE)
3,0.41,(CILANTRO)
1,0.41,(CEBOLLA)
8,0.38,(TORTILLAS DE MAÍZ)
4,0.36,(FRIJOL NEGRO)
0,0.35,(AGUACATE)
5,0.35,(LIMÓN)
9,0.33,(TOTOPOS)
31,0.33,"(TOMATE, CHILE JALAPEÑO)"


In [ ]:
# TAREA: Genera las reglas de asociación. Queremos reglas con un Lift mayor a 2. Almacena el resultado en la variable rules_mx.
### TU CÓDIGO AQUÍ ###
rules_mx = association_rules(frequent_itemsets_mx, metric="lift", min_threshold = 2)
rules_mx = rules_mx[['antecedents', 'consequents', 'antecedent support', 'consequent support', 'confidence', 'lift']]
rules_mx.sort_values(['lift', 'confidence'], ascending=[False, False]).head(15)

,antecedents,consequents,antecedent support,consequent support,confidence,lift
93,"(CEBOLLA, CHILE JALAPEÑO)","(CILANTRO, TOMATE)",0.32,0.32,0.92,2.90
88,"(CILANTRO, TOMATE)","(CEBOLLA, CHILE JALAPEÑO)",0.32,0.32,0.93,2.90
89,"(CILANTRO, CEBOLLA)","(TOMATE, CHILE JALAPEÑO)",0.31,0.33,0.95,2.88
92,"(TOMATE, CHILE JALAPEÑO)","(CILANTRO, CEBOLLA)",0.33,0.31,0.90,2.88
10,"(LIMÓN, AGUACATE)",(TOTOPOS),0.27,0.33,0.93,2.82
15,(TOTOPOS),"(LIMÓN, AGUACATE)",0.33,0.27,0.75,2.82
90,"(CILANTRO, CHILE JALAPEÑO)","(TOMATE, CEBOLLA)",0.32,0.33,0.92,2.81
91,"(TOMATE, CEBOLLA)","(CILANTRO, CHILE JALAPEÑO)",0.33,0.32,0.91,2.81
72,"(LIMÓN, AGUACATE, TOMATE)",(TOTOPOS),0.02,0.33,0.91,2.76
77,(TOTOPOS),"(LIMÓN, AGUACATE, TOMATE)",0.33,0.02,0.06,2.76


In [ ]:
# Ordena las reglas por Lift y Confianza de mayor a menor, muestra solamente las primeras 10 filas y las siguientes columnas:
# 'antecedents', 'consequents', 'antecedent support', 'consequent support', 'confidence', 'lift'
### TU CÓDIGO AQUÍ ###


3.3 Observa las 3 reglas con el Lift más alto para México (1, 3 y 5). **Interprétalas:** ¿Qué te dicen estas asociaciones? ¿Qué tipo de productos son?

Regla 1:

Las personas que compran cebolla y chile jalapeño tiene 92% de probabilidad de también llevar cilantro y tomate. El lift nos indica que las personas son 2.9 veces más propensas a comprar Cilantro y tomate.

Regla 3:

Las personas que compran cebolla y cilantro jalapeño tiene 95% de probabilidad de también llevar cilantro y tomate. El lift nos indica que las personas son 2.88 veces más propensas a comprar chile jalapeño y tomate.


Regla 5:

Las personas que compran limón y aguacate jalapeño tiene 93% de probabilidad de también llevar cilantro y tomate. El lift nos indica que las personas son 2.82 veces más propensas a comprar totopos.


Conclusión: Dado que todos tienen un lift superior al mínimo planteado de 2, todos los bienes son complementarios.


3.4 Para cada una de las 3 reglas (1, 3 y 5), interpreta el Soporte para el antecedente y el consecuente, la Confianza y el Lift

Regla 1:

El soporte del antecedente (0.32) indica que el 32% de todas las transacciones incluye cebolla y chile jalapeño, mientras que el soporte del consecuente (0.32) muestra que ese mismo porcentaje de compras contiene cilantro y tomate, lo que ya sugiere que ambos grupos de productos son bastante comunes. La confianza de 0.92 significa que, cuando un cliente compra cebolla y jalapeño, en el 92% de los casos también compra cilantro y tomate, evidenciando una relación muy fuerte. Finalmente, el lift de 2.90 indica que esta combinación ocurre casi tres veces más de lo esperado si las compras fueran independientes, confirmando una alta complementariedad entre estos ingredientes.

Regla 2:

El soporte del antecedente (0.31) muestra que el 31% de las transacciones incluye cilantro y cebolla, mientras que el soporte del consecuente (0.33) indica que el 33% contiene tomate y jalapeño, lo que refleja que ambos conjuntos son frecuentes en la canasta de compra. La confianza de 0.95 implica que, cuando se compran cilantro y cebolla, en el 95% de los casos también se adquieren tomate y jalapeño, lo que evidencia una relación aún más fuerte que en la regla anterior. El lift de 2.88 confirma que esta asociación es casi tres veces más probable de lo que sería por azar, reforzando la idea de que estos productos forman parte de una misma lógica de consumo.

Regla 3:

El soporte del antecedente (0.27) indica que el 27% de las transacciones incluye limón y aguacate, mientras que el soporte del consecuente (0.33) muestra que los totopos aparecen en el 33% de las compras, siendo un producto bastante frecuente. La confianza de 0.93 significa que, cuando un cliente compra limón y aguacate, en el 93% de los casos también compra totopos, lo que evidencia una relación muy sólida. Por último, el lift de 2.82 indica que esta combinación ocurre casi tres veces más de lo esperado bajo independencia, lo que sugiere una fuerte complementariedad asociada a un patrón de consumo específico, como la preparación de guacamole acompañado de totopos.

3.5 **Recomendación de Negocio:** Basado en estas reglas, ¿qué promoción o estrategia de venta específica podrías sugerir para el mercado mexicano?

1.Combos o Paquetes: Ofrecer combos que incluyan ingredientes que frecuentemente se compran juntos, como "Cebolla, Chile Jalapeño y Tomate". Esto no solo impulsa las ventas de estos productos, sino que también mejora la experiencia del cliente al facilitarles la compra de todo lo que necesitan para sus recetas.


2.Promociones Basadas en Combinaciones: Implementar descuentos al comprar pares de ingredientes que tienen alta confianza en las reglas, como "Limón y Aguacate" o "Tomate y Cebolla". Esto puede incentivar a los clientes a adquirir los productos que normalmente compran juntos.


3.Degustaciones en Tienda: Realizar degustaciones de platillos que incorporen fórmulas populares como "Totopos con Limón y Aguacate" en los puntos de venta. Esto puede estimular el interés del cliente y aumentar las ventas de esos ingredientes específicos.

3.6 Análisis para Colombia

In [ ]:
# TAREA: Usa la función preparar_cesta para obtener la matriz de transacciones de Colombia. Almacena el resultado en la variable cesta_co.
### TU CÓDIGO AQUÍ ###
cesta_co =  preparar_cesta(df_limpio,"Colombia")
cesta_co

Description,ACEITE DE GIRASOL,ARROZ,AZÚCAR,CAFÉ,FRIJOL CARGAMANTO,HARINA DE MAÍZ,HUEVOS,LECHE,PAN TAJADO,QUESO MUZZARELLA
InvoiceNo,,,,,,,,,,
536000,0,0,0,0,0,1,0,0,0,1
536001,0,0,0,0,0,1,0,0,0,1
536002,0,0,1,1,0,0,0,1,0,0
536003,0,0,0,0,0,0,1,0,1,0
536004,1,1,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
536995,0,0,1,1,0,1,0,0,0,1
536996,0,1,0,0,0,0,0,1,0,0
536997,1,1,0,0,1,0,0,0,0,0


In [ ]:
# TAREA: Aplica el algoritmo apriori con un soporte mínimo del 2%.
# Almacena el resultado en la variable frequent_itemsets_co.
# Muestra los 10 itemsets con el soporte más alto.
### TU CÓDIGO AQUÍ ###
frequent_itemsets_co = apriori(cesta_co, min_support = 0.02, use_colnames=True)
frequent_itemsets_co.sort_values(by='support', ascending=False).head(10)


,support,itemsets
3,0.41,(CAFÉ)
4,0.41,(FRIJOL CARGAMANTO)
0,0.40,(ACEITE DE GIRASOL)
2,0.40,(AZÚCAR)
7,0.39,(LECHE)
1,0.38,(ARROZ)
9,0.35,(QUESO MUZZARELLA)
5,0.34,(HARINA DE MAÍZ)
37,0.32,"(CAFÉ, LECHE)"
31,0.32,"(AZÚCAR, LECHE)"


In [ ]:
# TAREA: Genera las reglas de asociación con un Lift mayor a 2. Almacena el resultado en la variable rules_co.
### TU CÓDIGO AQUÍ ###
rules_co = association_rules(frequent_itemsets_co, metric="lift", min_threshold = 2)
rules_co = rules_co[['antecedents', 'consequents', 'antecedent support', 'consequent support', 'confidence', 'lift']]
rules_co.sort_values(['lift', 'confidence'], ascending=[False, False]).head(15)

,antecedents,consequents,antecedent support,consequent support,confidence,lift
40,"(AZÚCAR, FRIJOL CARGAMANTO, CAFÉ)",(LECHE),0.05,0.39,0.96,2.44
45,(LECHE),"(AZÚCAR, FRIJOL CARGAMANTO, CAFÉ)",0.39,0.05,0.11,2.44
10,"(AZÚCAR, CAFÉ)",(LECHE),0.32,0.39,0.95,2.41
15,(LECHE),"(AZÚCAR, CAFÉ)",0.39,0.32,0.76,2.41
5,"(ACEITE DE GIRASOL, FRIJOL CARGAMANTO)",(ARROZ),0.30,0.38,0.91,2.41
8,(ARROZ),"(ACEITE DE GIRASOL, FRIJOL CARGAMANTO)",0.38,0.30,0.73,2.41
58,"(AZÚCAR, PAN TAJADO, CAFÉ)",(LECHE),0.03,0.39,0.94,2.39
67,(LECHE),"(AZÚCAR, PAN TAJADO, CAFÉ)",0.39,0.03,0.07,2.39
48,"(AZÚCAR, HUEVOS, CAFÉ)",(LECHE),0.04,0.39,0.92,2.36
57,(LECHE),"(AZÚCAR, HUEVOS, CAFÉ)",0.39,0.04,0.09,2.36


In [ ]:
# Ordena las reglas por Lift y Confianza de mayor a menor, muestra solamente las primeras 10 filas y las siguientes columnas:
# 'antecedents', 'consequents', 'antecedent support', 'consequent support', 'confidence', 'lift'
### TU CÓDIGO AQUÍ ###


3.7 Observa las 3 reglas con el Lift más alto para Colombia (1, 3 y 5). **Interprétalas:** ¿Qué patrones de consumo específicos del mercado colombiano revelan estas reglas? ¿Son diferentes a las de México?

Regla 1:
Las personas que compran azúcar, frijol cargamanto y café tienen una probabilidad muy alta (96%) de también llevar leche. El lift indica que son 2.44 veces más propensas a comprar leche en comparación con lo esperado si las compras fueran independientes. Esto refleja un patrón de consumo típico colombiano asociado al desayuno y a la canasta básica del hogar.

Regla 3:
Las personas que compran azúcar y café tienen una probabilidad del 95% de también llevar leche. El lift muestra que son 2.41 veces más propensas a comprar leche, lo que evidencia una relación muy fuerte entre estos productos. Este patrón está directamente relacionado con el consumo habitual de café con leche en Colombia.

Regla 5:
Las personas que compran aceite de girasol y frijol cargamanto tienen una probabilidad del 91% de también llevar arroz. El lift de 2.41 indica que son más del doble de propensas a comprar arroz junto con estos productos, lo que refleja un patrón de consumo enfocado en la preparación de comidas tradicionales como arroz con frijoles.

Conclusión:
A diferencia de México, donde predominan combinaciones asociadas a consumo ocasional, en Colombia estas reglas muestran un consumo  diario y orientado a la alimentación del hogar con productos muy comunes en el desayuno y almuerzo. Dado que todos los lift son superiores a 2, se confirma una fuerte complementariedad entre los productos analizados.

3.8 Para cada una de las 3 reglas (1, 3 y 5), interpreta el Soporte para el antecedente y el consecuente, la Confianza y el Lift

Regla 1:
El soporte del antecedente (0.05) indica que el 5% de las transacciones incluye azúcar, frijol cargamanto y café, lo que muestra que es una combinación poco frecuente pero relevante. El soporte del consecuente (0.39) indica que la leche aparece en el 39% de las compras, siendo un producto muy común. La confianza de 0.96 implica que, cuando se da el antecedente, casi siempre se compra leche. El lift de 2.44 confirma que esta asociación es más del doble de probable que por azar, evidenciando una relación muy fuerte.

Regla 3:
El soporte del antecedente (0.32) muestra que el 32% de las transacciones incluye azúcar y café, siendo una combinación bastante común. El soporte del consecuente (0.39) indica nuevamente que la leche es un producto frecuente. La confianza de 0.95 señala que, en la gran mayoría de los casos, quienes compran azúcar y café también compran leche. El lift de 2.41 confirma una asociación fuerte y significativa entre estos productos.

Regla 5:
El soporte del antecedente (0.30) indica que el 30% de las transacciones incluye aceite de girasol y frijol cargamanto, lo que refleja una combinación frecuente. El soporte del consecuente (0.38) muestra que el arroz está presente en el 38% de las compras. La confianza de 0.91 implica una alta probabilidad de compra conjunta, y el lift de 2.41 confirma que esta combinación ocurre mucho más de lo esperado, evidenciando complementariedad en el consumo de productos básicos.

3.9 **Recomendación de Negocio:** ¿Qué campaña de marketing (diferente a la de México) podrías diseñar para los clientes colombianos?

1.Mercado rendidor para la casa:
Armar ofertas pensadas en el día a día de una familia, como juntar arroz, frijol y aceite o café, azúcar y leche con un pequeño descuento. La idea no es vender “combos llamativos”, sino facilitarle a la gente hacer un mercado completo y rendidor sin pensar mucho.

2.Ayuda al bolsillo:
Hacer promociones donde comprar más salga mejor, por ejemplo en arroz, leche o café. En Colombia mucha gente compra para toda la semana o quincena, entonces este tipo de ahorro se siente más útil que promociones tipo snack.

3.Recordatorios o recomendaciones simples:
Algo tan sencillo como sugerir “¿te falta la leche para el café?” o “lleva el arroz con los frijoles” en tienda o en la app. No es agresivo, pero sí ayuda a que la gente no olvide productos que normalmente compra juntos.